In [1]:
import sys
import os
import importlib

# Add the parent directory of utils.py to sys.path
sys.path.append(os.path.abspath("../src"))

import preprocess

importlib.reload(preprocess)

<module 'preprocess' from 'd:\\UrusanKuliah\\Perkuliahan\\Semester_7\\STKI\\stki-uts-A11202214623-StevenAdiSuryanto\\src\\preprocess.py'>

In [ ]:
from scipy.sparse import coo_matrix
import glob
import numpy as np
import re

### 1. Preprocess Data

In [3]:
all_words = []
all_files = []
file_folder = "../data/*.txt"
for file in glob.glob(file_folder):
    print(file)
    all_files.append(os.path.basename(file))

    file = open(file, "r", encoding="utf-8")
    text = file.read()

    cleaned_text = preprocess.clean(text)
    tokens = preprocess.tokenize(cleaned_text)
    tokens_no_stopwords = preprocess.remove_stopwords(tokens)
    tokens_stemmed = preprocess.stems(tokens_no_stopwords)
    all_words.extend(tokens_stemmed)

unique_all_words = set(all_words)

../data\cendrawasih.txt
../data\gajah_asia.txt
../data\harimau.txt
../data\katak.txt
../data\komodo.txt
../data\kuda_laut.txt
../data\orangutan.txt
../data\paus_biru.txt


In [4]:
unique_all_words = list(unique_all_words)
unique_all_words

['bulu',
 'peran',
 'ranting',
 'kilogram',
 'unik',
 'ingat',
 'nali',
 'air',
 'status',
 'lalat',
 'liput',
 'kera',
 'tropis',
 'manusia',
 'alam',
 'jangkrik',
 'komodo',
 'tajam',
 'liar',
 'mamalia',
 'endemik',
 'hilang',
 'jaga',
 'hijau',
 'pulau',
 'kilometer',
 'buru',
 'deforestasi',
 'soliter',
 'juta',
 'sendiri',
 'kerbau',
 'kuat',
 'gading',
 'habis',
 'habitat',
 'alat',
 'sabana',
 'kawin',
 'mangsa',
 'muda',
 'komunikasi',
 'populasi',
 'betina',
 'paus',
 'kenal',
 'oranye',
 'harimau',
 'racun',
 'suara',
 'predator',
 'asia',
 'tenggara',
 'pelihara',
 'kelompok',
 'indah',
 'selatan',
 'papua',
 'kalimantan',
 'kucing',
 'ajar',
 'daerah',
 'biru',
 'malam',
 'cenderawasih',
 'katak',
 'tangkar',
 'meter',
 'ratus',
 'ekosistem',
 'ancam',
 'panjang',
 'capai',
 'liur',
 'main',
 'makan',
 'besar',
 'utama',
 'hutan',
 'tahan',
 'warna',
 'kandung',
 'imbang',
 'licin',
 'babi',
 'pohon',
 'aktif',
 'ton',
 'simbol',
 'dunia',
 'bakteri',
 'hitam',
 'rusa',
 '

In [5]:
all_files

['cendrawasih.txt',
 'gajah_asia.txt',
 'harimau.txt',
 'katak.txt',
 'komodo.txt',
 'kuda_laut.txt',
 'orangutan.txt',
 'paus_biru.txt']

### 2a. Build incidence matrix

In [6]:
num_of_files = len(all_files)
num_of_words = len(unique_all_words)

row_positions = []
col_positions = []

for filename in all_files:
    filepath = "../data/" + filename
    file = open(filepath, "r", encoding="utf-8")
    # Preprocess
    text = file.read()
    cleaned_text = preprocess.clean(text)
    tokens = preprocess.tokenize(cleaned_text)
    tokens_no_stopwords = preprocess.remove_stopwords(tokens)
    tokens_stemmed = preprocess.stems(tokens_no_stopwords)
    tokens_final = set(tokens_stemmed)
    # Fill the positions
    for token in tokens_final:
        row_positions.append(unique_all_words.index(token))
        col_positions.append(all_files.index(filename))

data = [1] * len(row_positions)
incidence_matrix = coo_matrix(
    (data, (row_positions, col_positions)), shape=(num_of_words, num_of_files)
)

print(incidence_matrix.toarray())
print(incidence_matrix.shape)


[[1 0 1 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 0 ... 0 1 0]
 ...
 [0 0 0 ... 0 0 1]
 [0 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
(134, 8)


### 2b. Build inverted index

In [7]:
inverted_index = {}  # {'word': [(doc, freq, [positions])]}

for filename in all_files:
    filepath = "../data/" + filename
    file = open(filepath, "r", encoding="utf-8")
    # Preprocess
    text = file.read()
    cleaned_text = preprocess.clean(text)
    tokens = preprocess.tokenize(cleaned_text)
    tokens_no_stopwords = preprocess.remove_stopwords(tokens)
    tokens_stemmed = preprocess.stems(tokens_no_stopwords)

    position = {}  # {'word': [positions]}
    for index, word in enumerate(tokens_stemmed):
        if word in position:
            position[word].append(index)
        else:
            position[word] = [index]

    # Fill the index
    for word in position:
        if word in inverted_index:
            inverted_index[word].append(
                (all_files.index(filename), len(position[word]), position[word])
            )
        else:
            inverted_index[word] = [
                (all_files.index(filename), len(position[word]), position[word])
            ]
inverted_index

{'burung': [(0, 1, [0])],
 'cenderawasih': [(0, 1, [1])],
 'kenal': [(0, 1, [2]), (3, 1, [25]), (5, 1, [25])],
 'bulu': [(0, 1, [3]), (2, 1, [6])],
 'indah': [(0, 2, [4, 11])],
 'tari': [(0, 1, [5])],
 'kawin': [(0, 1, [6])],
 'unik': [(0, 1, [7])],
 'hidup': [(0, 1, [8]),
  (1, 2, [6, 15]),
  (2, 1, [3]),
  (3, 1, [9]),
  (5, 1, [9]),
  (6, 2, [5, 12]),
  (7, 1, [9])],
 'papua': [(0, 1, [9])],
 'simbol': [(0, 1, [10])],
 'alam': [(0, 1, [12])],
 'indonesia': [(0, 1, [13]), (4, 1, [5])],
 'gajah': [(1, 2, [0, 13])],
 'asia': [(1, 2, [1, 10]), (2, 1, [5])],
 'mamalia': [(1, 1, [2])],
 'darat': [(1, 1, [3])],
 'belalai': [(1, 1, [4])],
 'gading': [(1, 1, [5])],
 'hutan': [(1, 1, [7]),
  (2, 2, [4, 23]),
  (3, 1, [13]),
  (5, 1, [13]),
  (6, 1, [6])],
 'tropis': [(1, 1, [8])],
 'sabana': [(1, 1, [9])],
 'selatan': [(1, 1, [11])],
 'tenggara': [(1, 1, [12])],
 'sosial': [(1, 1, [14])],
 'kelompok': [(1, 1, [16])],
 'pimpin': [(1, 1, [17])],
 'betina': [(1, 1, [18])],
 'tua': [(1, 1, [19])]

### 3. Build parser query Boolean 

In [40]:
class BooleanQueryParser:
    def __init__(self, matrix: coo_matrix, terms: list[str], doc_ids: list[str]):
        self.matrix = matrix.tocsr()
        self.terms = terms
        self.term_index = {t: i for i, t in enumerate(terms)}
        self.doc_ids = doc_ids
        self.N = len(doc_ids)

    def _get_vector(self, term: str) -> np.ndarray:
        stemmed_term = "".join(preprocess.stems([term.lower()]))
        idx = self.term_index.get(stemmed_term)
        if idx is None:
            return np.zeros(self.N, dtype=np.int8)
        return (
            self.matrix[idx, :].toarray().ravel()
        )  # [0, 1, 1, ..., 0] untuk sebuah word

    def _not(self, vec):
        return 1 - vec

    def _and(self, v1, v2):
        return np.bitwise_and(v1, v2)

    def _or(self, v1, v2):
        return np.bitwise_or(v1, v2)

    def evaluate(self, query: str):
        tokens = re.findall(r"\(|\)|AND|OR|NOT|[A-Za-z0-9_]+", query.upper())

        # Convert to postfix (Shunting Yard)
        precedence = {"NOT": 3, "AND": 2, "OR": 1}
        output, stack = [], []

        for token in tokens:
            if token not in precedence:
                token = "".join(preprocess.stems([token.lower()]))
            if token in precedence:
                while (
                    stack
                    and stack[-1] != "("
                    and precedence[stack[-1]] >= precedence[token]
                ):
                    output.append(stack.pop())
                stack.append(token)
            elif token == "(":
                stack.append(token)
            elif token == ")":
                while stack and stack[-1] != "(":
                    output.append(stack.pop())
                stack.pop()
            else:
                output.append(token)

        while stack:
            output.append(stack.pop())

        # Evaluate postfix
        eval_stack = []
        for token in output:
            if token not in precedence:
                token = "".join(preprocess.stems([token.lower()]))
                eval_stack.append(self._get_vector(token))
            elif token == "NOT":
                v = eval_stack.pop()
                eval_stack.append(self._not(v))
            else:
                v2 = eval_stack.pop()
                v1 = eval_stack.pop()
                if token == "AND":
                    eval_stack.append(self._and(v1, v2))
                elif token == "OR":
                    eval_stack.append(self._or(v1, v2))

        result_vec = eval_stack.pop()
        return [self.doc_ids[i] for i, v in enumerate(result_vec) if v == 1]

In [46]:
parser = BooleanQueryParser(incidence_matrix, unique_all_words, all_files)

print(parser.evaluate("NOT serangga"))

['cendrawasih.txt', 'gajah_asia.txt', 'harimau.txt', 'komodo.txt', 'orangutan.txt', 'paus_biru.txt']


### 4. Evaluasi parser

In [48]:
queries = ["cendrawasih OR burung", "mangsa AND kerbau", "NOT serangga"]

gold_set = [
    ["cendrawasih.txt"],
    ["harimau.txt"],
    [
        "cendrawasih.txt",
        "gajah_asia.txt",
        "harimau.txt",
        "komodo.txt",
        "orangutan.txt",
        "paus_biru.txt",
    ],
]

parser = BooleanQueryParser(incidence_matrix, unique_all_words, all_files)
retrieval_result = [parser.evaluate(query) for query in queries]


def precision_at_k(retrieved, relevant):
    retrieved_set = set(retrieved)
    relevant_set = set(relevant)
    if not retrieved_set:
        return 0.0
    return len(retrieved_set & relevant_set) / len(retrieved_set)


for i, (retrieved, relevant) in enumerate(zip(retrieval_result, gold_set), 1):
    p = precision_at_k(retrieved, relevant)
    print(f"Query {i}:")
    print(f"  Precision: {p:.2f}")


Query 1:
  Precision: 1.00
Query 2:
  Precision: 1.00
Query 3:
  Precision: 1.00
